In [ ]:
import json
import sqlite3
import sys
import os
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import ipywidgets as widgets
from IPython.display import display, SVG

pio.renderers.default = "notebook_connected"

sys.path.insert(0, os.path.dirname(os.path.abspath("__file__")))

DB_PATH = os.path.join(os.path.dirname(os.path.abspath("__file__")), "positions.db")

In [ ]:
def load_positions(db_path: str) -> pd.DataFrame:
    conn = sqlite3.connect(db_path)
    rows = conn.execute("SELECT id, status, data, updated_at FROM positions").fetchall()
    conn.close()

    records = []
    for row_id, status, data_str, updated_at in rows:
        pos = json.loads(data_str)
        spreads = pos.get("spreads", {})
        features = pos.get("features", {})
        records.append({
            "id":             pos.get("id", row_id),
            "status":         status,
            "updated_at":     updated_at,
            "fen":            pos.get("fen"),
            "move_number":    pos.get("move_number"),
            "side_to_move":   pos.get("side_to_move"),
            "phase":          pos.get("phase"),
            "category":       pos.get("category"),
            "balance":        pos.get("balance"),
            "tag":            pos.get("tag"),
            "complexity":     ", ".join(pos.get("complexity", [])) or "none",
            "eco":            pos.get("eco"),
            "opening_name":   pos.get("opening_name"),
            "discard_reason": pos.get("discard_reason"),
            "best_cp":        spreads.get("best_cp"),
            "spread_1_2":     spreads.get("spread_1_2"),
            "spread_1_3":     spreads.get("spread_1_3"),
            "spread_1_5":     spreads.get("spread_1_5"),
            "spread_2_4":     spreads.get("spread_2_4"),
            "mobility":       features.get("mobility"),
            "captures":       features.get("captures"),
            "checks":         features.get("checks"),
            "blocked_pawns":  features.get("blocked_pawns"),
            "pawn_tension":   features.get("pawn_tension"),
            "_raw":           pos,
        })

    return pd.DataFrame(records)


df = load_positions(DB_PATH)
print(f"Loaded {len(df)} positions from {DB_PATH}")
print(df["status"].value_counts().to_string())

In [ ]:
fig = px.bar(
    df["status"].value_counts().reset_index(),
    x="status", y="count",
    title="Positions by Status",
    color="status",
)
fig.show()

In [ ]:
passed = df[df["status"] == "fine_filter_passed"].copy()

w_phase    = widgets.SelectMultiple(options=["(all)"] + sorted(passed["phase"].dropna().unique().tolist()),    value=["(all)"], description="Phase",    rows=5)
w_balance  = widgets.SelectMultiple(options=["(all)"] + sorted(passed["balance"].dropna().unique().tolist()),  value=["(all)"], description="Balance",  rows=6)
w_category = widgets.SelectMultiple(options=["(all)"] + sorted(passed["category"].dropna().unique().tolist()), value=["(all)"], description="Category", rows=6)
w_tag      = widgets.SelectMultiple(options=["(all)"] + sorted(passed["tag"].dropna().unique().tolist()),      value=["(all)"], description="Tag",      rows=4)

out = widgets.Output()

def apply_filters(phase_sel, balance_sel, category_sel, tag_sel):
    filt = passed.copy()
    if "(all)" not in phase_sel:    filt = filt[filt["phase"].isin(phase_sel)]
    if "(all)" not in balance_sel:  filt = filt[filt["balance"].isin(balance_sel)]
    if "(all)" not in category_sel: filt = filt[filt["category"].isin(category_sel)]
    if "(all)" not in tag_sel:      filt = filt[filt["tag"].isin(tag_sel)]
    return filt

def on_change(_):
    filt = apply_filters(w_phase.value, w_balance.value, w_category.value, w_tag.value)
    with out:
        out.clear_output(wait=True)
        print(f"{len(filt)} positions match")

        px.histogram(filt, x="move_number", nbins=30, title="Move Number Distribution", color="phase").show()
        px.histogram(filt, x="best_cp", nbins=40, title="Best CP Distribution", color="balance").show()
        px.histogram(filt, x="spread_1_2", nbins=40, title="Spread 1-2 (cp)").show()
        px.scatter(filt, x="mobility", y="captures", color="phase",
                   hover_data=["id", "balance", "category"],
                   title="Mobility vs Captures", opacity=0.6).show()
        px.scatter(filt, x="spread_1_2", y="best_cp", color="category",
                   hover_data=["id", "balance", "phase"],
                   title="Spread 1-2 vs Best CP", opacity=0.6).show()

        complexity_counts = filt["complexity"].str.split(", ").explode().value_counts().reset_index()
        px.bar(complexity_counts, x="complexity", y="count", title="Complexity Tags").show()

        px.pie(filt, names="phase", title="Phase").show()

        balance_order = ["winning", "better", "equal", "worse", "losing"]
        balance_counts = filt["balance"].value_counts().reindex(balance_order).dropna().reset_index()
        px.bar(balance_counts, x="balance", y="count", title="Balance", color="balance").show()

for w in [w_phase, w_balance, w_category, w_tag]:
    w.observe(on_change, names="value")

display(widgets.HBox([w_phase, w_balance, w_category, w_tag]), out)
on_change(None)

In [ ]:
failed = df[df["status"] == "fine_filter_failed"].copy()
if not failed.empty:
    px.bar(
        failed["discard_reason"].value_counts().reset_index(),
        x="discard_reason", y="count",
        title=f"Discard Reasons ({len(failed)} positions)",
        color="discard_reason",
    ).show()
else:
    print("No failed positions in DB")

In [ ]:
eco_df = (
    passed.dropna(subset=["eco"])
    .groupby(["eco", "opening_name"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
    .head(20)
)
eco_df["label"] = eco_df["eco"] + " " + eco_df["opening_name"].fillna("")
fig = px.bar(eco_df, x="label", y="count", title="Top 20 Openings (passed positions)")
fig.update_layout(xaxis_tickangle=-40)
fig.show()

In [ ]:
try:
    import chess
    import chess.svg
    HAS_CHESS = True
except ImportError:
    HAS_CHESS = False
    print("pip install chess  to enable board rendering")

id_options = passed["id"].tolist()

w_id = widgets.Combobox(
    options=id_options,
    value=id_options[0] if id_options else "",
    description="Position ID:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="420px"),
)
out_inspect = widgets.Output()

def show_position(_):
    pos_id = w_id.value
    rows = passed[passed["id"] == pos_id]
    if rows.empty:
        with out_inspect:
            out_inspect.clear_output()
            print(f"ID '{pos_id}' not found")
        return

    row = rows.iloc[0]
    raw = row["_raw"]

    with out_inspect:
        out_inspect.clear_output(wait=True)

        if HAS_CHESS:
            board = chess.Board(row["fen"])
            svg = chess.svg.board(board, size=350, orientation=(row["side_to_move"] == "white"))
            display(SVG(svg))

        print(f"ID:          {row['id']}")
        print(f"FEN:         {row['fen']}")
        print(f"Side:        {row['side_to_move']}  move {row['move_number']}")
        print(f"Phase:       {row['phase']}")
        print(f"Balance:     {row['balance']}  ({row['best_cp']} cp)")
        print(f"Category:    {row['category']}")
        print(f"Complexity:  {row['complexity']}")
        print(f"Tag:         {row['tag']}")
        print(f"ECO:         {row['eco']}  {row['opening_name']}")
        print(f"Features:    mobility={row['mobility']}  captures={row['captures']}  checks={row['checks']}")
        print(f"Spreads:     1-2={row['spread_1_2']}  1-3={row['spread_1_3']}  1-5={row['spread_1_5']}")

        best_eval = max(raw.get("evals", []), key=lambda e: e.get("depth", 0), default={})
        pvs = best_eval.get("pvs", [])
        if pvs:
            print(f"\nTop PVs (depth {best_eval.get('depth', '?')}):")
            for i, pv in enumerate(pvs[:5], 1):
                score = f"cp={pv['cp']}" if "cp" in pv else f"mate={pv.get('mate')}"
                moves = " ".join(pv.get("moves", [])[:5])
                print(f"  {i}. {score:>12}   {moves}")

w_id.observe(show_position, names="value")
display(w_id, out_inspect)
show_position(None)